import logging
import os
import sys
from io import BytesIO
import pandas as pd
import requests
from datetime import date, timedelta, datetime
from collections import Counter
import numpy as np
import json
from astroquery.vizier import Vizier
import astropy.units as u
from astropy.io import fits
from astropy.coordinates import SkyCoord
from tqdm import tqdm
import time
from astropy.table import Table, Column, vstack, join
import re
from astropy.time import Time
from glob import glob
from multiprocessing import Pool

# Project-Specific Imports
from desispec.io import read_spectra
from desitrip.preproc import rebin_flux, rescale_flux
from desispec.coaddition import coadd_cameras
from desiutil.depend import Dependencies
from scipy.ndimage import gaussian_filter1d

In [2]:
def load_metadata_table(file_path):
    """Load the FASTSPEC table from a FITS file."""
    with fits.open(file_path) as hdul:
        return Table(hdul['METADATA'].data)

In [4]:
loa_path = "/global/cfs/cdirs/desi/vac/dr2/fastspecfit/loa/v1.0/catalogs"
fastspec_files = np.sort(glob(os.path.join(loa_path, "*.fits")))
loa_metadata = vstack([load_metadata_table(os.path.join(loa_path, i)) for i in fastspec_files])

In [5]:
df = pd.read_csv(
        "tns_public_objects.csv.zip",
        #names = col_names,
        skiprows = 1,
        compression='zip',
        low_memory=False
    )
df["discoverydate"] = pd.to_datetime(df["discoverydate"])

In [6]:
loa_coords = SkyCoord(ra=loa_metadata['RA'], dec=loa_metadata['DEC'], unit=(u.deg, u.deg))
tns_coords = SkyCoord(ra=df['ra'], dec=df['declination'], unit=(u.deg, u.deg))

In [7]:
max_sep = 1 * u.arcsec

In [8]:
df_smaller = Table.from_pandas(df[["name_prefix", "name", "type", "discoverydate"]])

In [9]:
idx, sep2d, _ = tns_coords.match_to_catalog_sky(loa_coords)

matched = sep2d <= max_sep

merged_data = df_smaller.copy()

# Add columns from table2 where matches are found
# For simplicity, we'll add 'id2' from table2; you can add more columns as needed
merged_data['TARGETID'] = [-1] * len(df_smaller)  # Initialize with empty strings
#merged_data['SPECTYPE'] = ["GALAXY"] * len(df_smaller)  # Initialize with empty strings

# Assign 'id2' where matches are found
merged_data['TARGETID'][matched] = loa_metadata['TARGETID'][idx[matched]]
#merged_data['SPECTYPE'][matched] = loa_metadata["SPECTYPE"][idx[matched]]

# Optionally, add the separation as a new column
merged_data['separation_arcsec'] = sep2d.arcsecond

desi_data = merged_data[merged_data["TARGETID"] != -1]
desi_data

name_prefix,name,type,discoverydate,TARGETID,separation_arcsec
str3,str9,str17,Time,int64,float64
AT,2025hbi,--,2025-04-01T01:31:27.264,39627883848666994,0.04643662767978609
AT,2025hbg,--,2025-04-01T07:04:16.896,39627806472148261,0.6946461316995944
AT,2025hba,--,2025-02-15T07:25:27.840,39627776235407424,0.6190181942891215
AT,2025hay,--,2025-04-01T02:04:58.656,39627618613463237,0.6650496732862585
AT,2025haw,--,2025-03-29T06:51:31.392,39627860687719890,0.543080417789161
AT,2025hav,--,2025-03-29T07:06:32.544,39627842564133178,0.43866249814285324
AT,2025has,--,2025-03-26T06:58:31.296,39627697655124640,0.7171748138080044
AT,2025har,--,2025-03-26T06:45:48.384,39627721793340566,0.7474972671592723
AT,2025hao,--,2025-03-26T04:21:50.112,39627660715888121,0.45692903832671206


In [10]:
tns_desi_data = join(desi_data, loa_metadata, keys='TARGETID', join_type='inner')

In [11]:
loa_directory = "/global/cfs/cdirs/desi/spectro/redux/loa/tiles/pernight"
loa_cumulative_directory = "/global/cfs/cdirs/desi/spectro/redux/loa/tiles/cumulative"

In [12]:
del loa_metadata
import gc
gc.collect()

0

In [13]:
tns_desi_data

name_prefix,name,type,discoverydate,TARGETID,separation_arcsec,SURVEY,PROGRAM,HEALPIX,TILEID_LIST,RA,DEC,COADD_FIBERSTATUS,CMX_TARGET,DESI_TARGET,BGS_TARGET,MWS_TARGET,SCND_TARGET,SV1_DESI_TARGET,SV1_BGS_TARGET,SV1_MWS_TARGET,SV2_DESI_TARGET,SV2_BGS_TARGET,SV2_MWS_TARGET,SV3_DESI_TARGET,SV3_BGS_TARGET,SV3_MWS_TARGET,SV1_SCND_TARGET,SV2_SCND_TARGET,SV3_SCND_TARGET,Z,ZWARN,DELTACHI2,SPECTYPE,SUBTYPE,Z_RR,ZWARN_RR,TSNR2_BGS,TSNR2_LRG,TSNR2_ELG,TSNR2_QSO,TSNR2_LYA,PHOTSYS,LS_ID,FIBERFLUX_G,FIBERFLUX_R,FIBERFLUX_Z,FIBERTOTFLUX_G,FIBERTOTFLUX_R,FIBERTOTFLUX_Z,FLUX_G,FLUX_R,FLUX_Z,FLUX_W1,FLUX_W2,FLUX_W3,FLUX_W4,FLUX_IVAR_G,FLUX_IVAR_R,FLUX_IVAR_Z,FLUX_IVAR_W1,FLUX_IVAR_W2,FLUX_IVAR_W3,FLUX_IVAR_W4,EBV,MW_TRANSMISSION_G,MW_TRANSMISSION_R,MW_TRANSMISSION_Z,MW_TRANSMISSION_W1,MW_TRANSMISSION_W2,MW_TRANSMISSION_W3,MW_TRANSMISSION_W4
str3,str9,str17,Time,int64,float64,str7,str6,int32,str365,float64,float64,int32,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,float64,int64,float64,str6,str20,float64,int64,float32,float32,float32,float32,float32,str1,int64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float32,float32,float32,float32,float32,float32,float32
AT,2017kin,--,2017-03-29T06:57:36.000,142180039524356,0.216081365029569,sv1,other,27257,80871,150.79597,2.16742,0,0,0,0,0,0,4611686018427387904,0,0,0,0,0,0,0,0,4294967296,0,0,1.476346674757886,0,64.0109811425209,GALAXY,,1.476346674757886,0,130923.16,1448.2932,2141.7395,568.81116,2230.3345,S,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.018982543579598672,0.94539046,0.9628723,0.97905,0.9967882,0.9980263,0.9995787,0.9998409
AT,2016fwe,--,2016-08-17T13:55:12.000,2253071405023237,0.6234894044684527,main,dark,17692,4803,36.55810518333876,-4.67678939773213,0,--,4611686018427387904,0,0,4096,--,--,--,--,--,--,--,--,--,--,--,--,0.05447493606324368,0,3550.9801465272903,GALAXY,,0.05447493606324368,0,8258.579,96.488495,155.1676,38.71002,115.78697,S,9906619634883119,0.09662779,0.23489255,0.07467852,0.10460051,0.24778028,0.07694445,0.29820573,0.7063971,0.21936104,0.4324536,-0.6168442,-30.869663,439.28418,1168.8489,632.4291,69.04189,3.1835759,0.702796,0.0015140816,2.1201788e-05,0.026799337006702673,0.92377925,0.94798726,0.9705511,0.9954686,0.9972147,0.9994053,0.9997754
SN,2018hse,SLSN-II,2018-10-06T09:28:48.000,2323290886504448,0.0767924405360447,main,dark,36343,2449,57.5261672801719,-11.0034323294605,0,--,4611686018427387904,0,0,268435456,--,--,--,--,--,--,--,--,--,--,--,--,0.1295328386839476,0,1205.3010593652725,GALAXY,,0.1295328386839476,0,8569.458,105.25522,162.18689,41.907967,113.16844,S,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.041682902993952456,0.88398683,0.9202785,0.95457226,0.9929609,0.99567115,0.9990752,0.9996507
AT,2024fuc,--,2024-04-03T19:25:18.336,2349890482995200,0.8130948934169695,main,dark,27273,4089,139.68076171066753,-2.335388738686538,0,--,4611686018427387904,0,0,17179869184,--,--,--,--,--,--,--,--,--,--,--,--,0.4602167589484957,0,735.6139181889594,GALAXY,,0.4602167589484957,0,7355.6636,86.21625,122.80042,34.31633,143.74648,S,9906620604485627,2.5503864,6.5022507,14.416077,2.8424222,6.994946,15.017524,5.3518515,13.170426,28.275728,62.249126,48.608284,-35.14278,-2085.2812,408.42285,127.88119,42.178867,1.8126879,0.5341494,0.0008901577,1.15330095e-05,0.03664585533140164,0.89725816,0.9295641,0.95995027,0.99380887,0.9961933,0.99918693,0.9996929
AT,2018brs,--,2018-04-06T16:52:54.000,2389329557061635,0.9843523026774197,main,bright,25793,23725,203.67096699494707,-8.23507334148321,0,--,4611686018427387904,0,0,8796093022208,--,--,--,--,--,--,--,--,--,--,--,--,0.10223792472491368,0,595.0430281758308,GALAXY,,0.10223792472491368,0,1898.324,22.392565,36.526855,7.1363163,8.976477,S,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

In [14]:
def process_row(row):
    results = []
    tileid_path = os.path.join(loa_directory, row["TILEID_LIST"])
    tileid_cum_path = os.path.join(loa_cumulative_directory, row["TILEID_LIST"])
    if os.path.exists(tileid_path):
        date_list = os.listdir(tileid_path)
        for date in date_list:
            data_path = os.path.join(tileid_path, date)
            file_list = np.sort(glob(os.path.join(data_path, "coadd-*.fits")))
            data_info = [read_spectra(i, targetids=row["TARGETID"]) for i in file_list]
            spectra_list = [d for d in data_info if d.fibermap is not None]
            if spectra_list:
                spectra = spectra_list[0]
                results.append((row["TARGETID"], date, spectra))
    elif os.path.exists(tileid_cum_path):
        date_list = os.listdir(tileid_cum_path)
        for date in date_list:
            data_path = os.path.join(tileid_cum_path, date)
            file_list = np.sort(glob(os.path.join(data_path, "coadd-*.fits")))
            data_info = [read_spectra(i, targetids=row["TARGETID"]) for i in file_list]
            spectra_list = [d for d in data_info if d.fibermap is not None]
            if spectra_list:
                spectra = spectra_list[0]
                results.append((row["TARGETID"], date, spectra))
    else:
        return "missing"
    return results

In [15]:
with Pool(128) as pool:
    all_results = list(tqdm(pool.imap(process_row, tns_desi_data), total=len(tns_desi_data)))

# Flatten and separate
desi_targetids, desi_night, desi_spectra = [], [], []
counter = 0

for item in all_results:
    if item == "missing":
        counter += 1
    else:
        for targetid, night, spectrum in item:
            desi_targetids.append(targetid)
            desi_night.append(night)
            desi_spectra.append(spectrum)

print(f"Missing counter: {counter}")

  0%|          | 0/17599 [00:00<?, ?it/s]

INFO:spectra.py:451:read_spectra: iotime 0.976 sec to read spectra from:  coadd-0-4089-thru20211106.fits at 2025-06-18T22:34:33.350012
INFO:spectra.py:451:read_spectra: iotime 1.031 sec to read spectra from:  coadd-2-4803-thru20230906.fits at 2025-06-18T22:34:33.877194
INFO:spectra.py:451:read_spectra: iotime 0.870 sec to read spectra from:  coadd-3-2449-thru20221220.fits at 2025-06-18T22:34:34.097176
INFO:spectra.py:451:read_spectra: iotime 0.976 sec to read spectra from:  coadd-2-23083-thru20230226.fits at 2025-06-18T22:34:34.227851
INFO:spectra.py:451:read_spectra: iotime 0.694 sec to read spectra from:  coadd-1-21132-thru20230415.fits at 2025-06-18T22:34:34.433275
INFO:spectra.py:451:read_spectra: iotime 0.984 sec to read spectra from:  coadd-0-25484-thru20240327.fits at 2025-06-18T22:34:35.124374
INFO:spectra.py:451:read_spectra: iotime 0.910 sec to read spectra from:  coadd-2-22470-thru20220510.fits at 2025-06-18T22:34:35.387076
INFO:spectra.py:451:read_spectra: iotime 0.985 sec 

  0%|          | 1/17599 [00:10<49:07:56, 10.05s/it]

INFO:spectra.py:451:read_spectra: iotime 1.051 sec to read spectra from:  coadd-2-22381-thru20220612.fits at 2025-06-18T22:34:42.067109
INFO:spectra.py:451:read_spectra: iotime 1.016 sec to read spectra from:  coadd-4-4119-thru20240115.fits at 2025-06-18T22:34:42.102325
INFO:spectra.py:451:read_spectra: iotime 1.036 sec to read spectra from:  coadd-4-23318-thru20211219.fits at 2025-06-18T22:34:42.149181
INFO:spectra.py:451:read_spectra: iotime 5.716 sec to read spectra from:  coadd-0-6558-thru20230824.fits at 2025-06-18T22:34:42.163365
INFO:spectra.py:451:read_spectra: iotime 1.036 sec to read spectra from:  coadd-6-11424-thru20231004.fits at 2025-06-18T22:34:42.173255
INFO:spectra.py:451:read_spectra: iotime 1.120 sec to read spectra from:  coadd-1-21914-thru20240113.fits at 2025-06-18T22:34:42.304521
INFO:spectra.py:451:read_spectra: iotime 1.085 sec to read spectra from:  coadd-8-3595-thru20221001.fits at 2025-06-18T22:34:42.327104
INFO:spectra.py:451:read_spectra: iotime 1.157 sec 

  0%|          | 5/17599 [00:22<20:14:55,  4.14s/it]

...

INFO:spectra.py:451:read_spectra: iotime 1.106 sec to read spectra from:  coadd-6-41147-thru20240219.fits at 2025-06-18T22:52:22.358962
INFO:spectra.py:451:read_spectra: iotime 30.225 sec to read spectra from:  coadd-9-41924-thru20230504.fits at 2025-06-18T22:52:24.789363


 99%|█████████▊| 17372/17599 [17:52<00:19, 11.63it/s]

INFO:spectra.py:451:read_spectra: iotime 18.202 sec to read spectra from:  coadd-7-42776-thru20231221.fits at 2025-06-18T22:52:32.768286


100%|██████████| 17599/17599 [18:01<00:00, 16.27it/s]


Missing counter: 1356


In [16]:
print(f"Missing counter: {counter}")

Missing counter: 1356


In [17]:
desi_tns_table = Table()
desi_tns_table["TARGETID"] = np.asarray(desi_targetids)
desi_tns_table["NIGHT"] = np.asarray(desi_night)

In [18]:
for tid, night, spec in tqdm(zip(desi_targetids, desi_night, desi_spectra)):
    spectra = coadd_cameras(spec)

    smoothed_flux = gaussian_filter1d(spectra.flux["brz"][0], 3)
    ivar_numeric = np.where(spectra.ivar["brz"] > 0, 1 / spectra.ivar["brz"], np.nan)  # Replace invalid entries with NaN
    std = np.sqrt(ivar_numeric)
    smoothed_std = gaussian_filter1d(std, 3)
    smoothed_std_clean = np.nan_to_num(smoothed_std, nan=0.0, posinf=0.0, neginf=0.0)

    spectrum_table = Table()
    spectrum_table['wavelength'] = spectra.wave["brz"]
    spectrum_table['flux'] = smoothed_flux
    spectrum_table['fluxerr'] = smoothed_std_clean[0]

    spectrum_table.to_pandas().to_csv("/global/cfs/cdirs/m4237/xhall/data/loa_spectra/{}_{}.csv".format(tid, night))

/tmp/ipykernel_358348/3093989029.py:5: RuntimeWarning: divide by zero encountered in true_divide
  ivar_numeric = np.where(spectra.ivar["brz"] > 0, 1 / spectra.ivar["brz"], np.nan)  # Replace invalid entries with NaN
16243it [35:50,  7.55it/s]


In [19]:
output_spectra_table = join(tns_desi_data, desi_tns_table, keys='TARGETID', join_type='inner')

In [20]:
output_spectra_df = output_spectra_table.to_pandas()

In [21]:
output_spectra_df["obsdate"] = pd.to_datetime(output_spectra_df["NIGHT"].astype(int).astype(str), format="%Y%m%d")

In [22]:
output_spectra_df['days_since_discovery'] = (output_spectra_df['obsdate'] - output_spectra_df['discoverydate']).dt.days

In [23]:
output_spectra_df.to_csv("desi_tns_spectra_loa.csv", index = False)

In [24]:
output_spectra_df.loc[(output_spectra_df["days_since_discovery"] < 120) & (output_spectra_df["days_since_discovery"] > -30)].to_csv("desi_tns_spectra_transients_loa.csv", index = False)

In [27]:
output_spectra_df

,name_prefix,name,type,discoverydate,TARGETID,separation_arcsec,SURVEY,PROGRAM,HEALPIX,TILEID_LIST,...,MW_TRANSMISSION_G,MW_TRANSMISSION_R,MW_TRANSMISSION_Z,MW_TRANSMISSION_W1,MW_TRANSMISSION_W2,MW_TRANSMISSION_W3,MW_TRANSMISSION_W4,NIGHT,obsdate,days_since_discovery
0,AT,2017kin,NaN,2017-03-29 06:57:36.000,142180039524356,0.216081,sv1,other,27257,80871,...,0.945390,0.962872,0.979050,0.996788,0.998026,0.999579,0.999841,20210430,2021-04-30,1492
1,AT,2016fwe,NaN,2016-08-17 13:55:12.000,2253071405023237,0.623489,main,dark,17692,4803,...,0.923779,0.947987,0.970551,0.995469,0.997215,0.999405,0.999775,20230906,2023-09-06,2575
2,SN,2018hse,SLSN-II,2018-10-06 09:28:48.000,2323290886504448,0.076792,main,dark,36343,2449,...,0.883987,0.920278,0.954572,0.992961,0.995671,0.999075,0.999651,20221220,2022-12-20,1535
3,AT,2024fuc,NaN,2024-04-03 19:25:18.336,2349890482995200,0.813095,main,dark,27273,4089,...,0.897258,0.929564,0.959950,0.993809,0.996193,0.999187,0.999693,20211106,2021-11-06,-880
4,AT,2018brs,NaN,2018-04-06 16:52:54.000,2389329557061635,0.984352,main,bright,25793,23725,...,0.888658,0.923552,0.956471,0.993261,0.995856,0.999115,0.999666,20230401,2023-04-01,1820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18551,AT,2024vzg,NaN,2024-09-17 11:24:00.000,2305843047847432574,0.102453,main,backup,36846,41677,...,0.917604,0.943713,0.968100,0.995086,0.996979,0.999355,0.999756,20230826,2023-08-26,-389
18552,AT,2019oka,NaN,2019-08-23 10:49:54.000,2305843047860012276,0.321537,main,backup,36856,41687,...,0.860010,0.903386,0.944727,0.991398,0.994708,0.998869,0.999573,20211220,2021-12-20,849
18553,AT,2022hzb,NaN,2022-04-05 04:30:43.200,2305843051370683613,0.234682,main,backup,40204,40363,...,0.773572,0.841164,0.907742,0.985400,0.991009,0.998076,0.999273,20230423,2023-04-23,382
18554,AT,2022kag,NaN,2022-04-23 12:40:19.200,2305843052104650007,0.047339,main,backup,40907,43078,...,0.913195,0.940656,0.966343,0.994812,0.996810,0.999319,0.999743,20230209,2023-02-09,291


In [28]:
print(output_spectra_df["type"].value_counts())

type
SN Ia                1022
SN II                 178
TDE                    55
AGN                    44
SN Ia-91T-like         43
SN IIn                 42
SN Ic                  31
SN IIb                 20
Galaxy                 20
SN Ib                  15
SN Ic-BL               15
SN IIP                 14
SN I                   13
SN Ia-pec              12
SLSN-II                10
SLSN-I                  6
SN Ia-CSM               4
SN                      4
CV                      4
SN Ib/c                 4
TDE-He                  4
SN Ibn                  4
QSO                     3
SN Ia-91bg-like         2
Blazar                  2
Nova                    2
SN II-pec               1
SN Ib-pec               1
TDE-H-He                1
LBV                     1
SN IIL                  1
SN Iax[02cx-like]       1
Name: count, dtype: int64
